# RAG 入门指南

尽管大型语言模型（LLM）展现出强大的能力并支撑着许多高级应用场景，但它们也存在事实不一致和幻觉等问题。检索增强生成（RAG）是一种强大的方法，可以丰富 LLM 的能力并提高其可靠性。RAG  通过将 LLM 与外部知识相结合来实现这一目标，具体方式是在提示词中加入相关信息作为上下文，帮助完成特定任务。

本教程展示如何通过利用向量数据库和开源 LLM 来入门 RAG。为了展示 RAG 的强大功能，本用例将构建一个 RAG 系统，用于根据原始 ML 论文标题生成简短且易于阅读的论文标题。论文标题可能对普通读者来说过于专业，因此可以使用 RAG 基于先前创建的简短标题来生成简短标题，使科研论文标题更加通俗易懂，可用于科学传播，如Newsletter或博客文章。

在开始之前，让我们先安装所需的库：

In [ ]:
%%capture
!pip install chromadb tqdm fireworks-ai python-dotenv pandas
!pip install sentence-transformers

在继续之前，你需要获取一个 Fireworks API Key 来使用 Mistral 7B 模型。

获取 Fireworks API Key 的快速指南：https://readme.fireworks.ai/docs

In [1]:
import fireworks.client
import os
import dotenv
import chromadb
import json
from tqdm.auto import tqdm
import pandas as pd
import random

# 可以使用 Colab secrets 设置环境变量
dotenv.load_dotenv()

fireworks.client.api_key = os.getenv("FIREWORKS_API_KEY")

/home/fnliren/Downloads/miniconda3/envs/pe-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 入门指南

让我们定义一个函数来从 Fireworks 推理平台获取补全结果。

In [ ]:
def get_completion(prompt, model=None, max_tokens=50):

    fw_model_dir = "accounts/fireworks/models/"

    if model is None:
        model = fw_model_dir + "gpt-oss-20b"
    else:
        model = fw_model_dir + model

    completion = fireworks.client.Completion.create(
        model=model,
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=0
    )

    return completion.choices[0].text

让我们先用简单的提示词测试一下这个函数：

In [3]:
get_completion("Hello, my name is")

InvalidRequestError: {"error": {"message": "Model not found, inaccessible, and/or not deployed", "param": "model", "code": "NOT_FOUND", "type": "error"}, "request_id": "cmpl-66c31a8da51c48edade58eeea7305785"}

现在让我们用 Mistral-7B-Instruct 测试：

In [ ]:
mistral_llm = "mistral-7b-instruct-4k"

get_completion("Hello, my name is", model=mistral_llm)

Mistral 7B Instruct 模型需要使用特殊的指令标记 `[INST] <instruction> [/INST]` 来获得正确的行为。你可以在以下链接找到更多关于如何提示 Mistral 7B Instruct 的说明：https://docs.mistral.ai/llm/mistral-instruct-v0.1

In [ ]:
mistral_llm = "mistral-7b-instruct-4k"

get_completion("Tell me 2 jokes", model=mistral_llm)

In [ ]:
mistral_llm = "mistral-7b-instruct-4k"

get_completion("[INST]Tell me 2 jokes[/INST]", model=mistral_llm)

现在让我们用更复杂的包含指令的提示词试试：

In [ ]:
prompt = """[INST]
Given the following wedding guest data, write a very short 3-sentences thank you letter:

{
  "name": "John Doe",
  "relationship": "Bride's cousin",
  "hometown": "New York, NY",
  "fun_fact": "Climbed Mount Everest in 2020",
  "attending_with": "Sophia Smith",
  "bride_groom_name": "Tom and Mary"
}

Use only the data provided in the JSON object above.

The senders of the letter is the bride and groom, Tom and Mary.
[/INST]"""

get_completion(prompt, model=mistral_llm, max_tokens=150)

## RAG 用例：生成简短论文标题

在 RAG 用例中，我们将使用一个包含每周热门 ML 论文列表的数据集。

用户将提供一个原始论文标题。然后我们将使用该数据集生成简短且吸引人的论文标题上下文，帮助为原始标题生成吸引人的标题。

### 第一步：加载数据集

首先让我们加载要使用的数据集：

In [ ]:
# 从 data/ 文件夹加载数据集到 pandas DataFrame
# 数据集包含列名

ml_papers = pd.read_csv("../data/ml-potw-10232023.csv", header=0)

# 移除标题或描述为空的行
ml_papers = ml_papers.dropna(subset=["Title", "Description"])

In [ ]:
ml_papers.head()

In [ ]:
# 将 DataFrame 转换为字典列表，只包含 Title 和 Description 列

ml_papers_dict = ml_papers.to_dict(orient="records")

In [ ]:
ml_papers_dict[0]

我们将使用 SentenceTransformer 生成嵌入向量，存储到 Chroma 文档数据库中。

In [ ]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

class MyEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        batch_embeddings = embedding_model.encode(input)
        return batch_embeddings.tolist()

embed_fn = MyEmbeddingFunction()

# 初始化 chromadb 目录和客户端
client = chromadb.PersistentClient(path="./chromadb")

# 创建 collection
collection = client.get_or_create_collection(
    name=f"ml-papers-nov-2023"
)

现在让我们批量生成嵌入向量：

In [ ]:
# 批量生成嵌入向量并索引标题
batch_size = 50

# 遍历批量数据，生成并存储嵌入向量
for i in tqdm(range(0, len(ml_papers_dict), batch_size)):

    i_end = min(i + batch_size, len(ml_papers_dict))
    batch = ml_papers_dict[i : i + batch_size]

    # 如果标题为空字符串则替换为 "No Title"
    batch_titles = [str(paper["Title"]) if str(paper["Title"]) != "" else "No Title" for paper in batch]
    batch_ids = [str(sum(ord(c) + random.randint(1, 10000) for c in paper["Title"])) for paper in batch]
    batch_metadata = [dict(url=paper["PaperURL"],
                           abstract=paper['Abstract'])
                           for paper in batch]

    # 生成嵌入向量
    batch_embeddings = embedding_model.encode(batch_titles)

    #  upsert 到 chromadb
    collection.upsert(
        ids=batch_ids,
        metadatas=batch_metadata,
        documents=batch_titles,
        embeddings=batch_embeddings.tolist(),
    )

现在我们可以测试检索器：

In [ ]:
collection = client.get_or_create_collection(
    name=f"ml-papers-nov-2023",
    embedding_function=embed_fn
)

retriever_results = collection.query(
    query_texts=["Software Engineering"],
    n_results=2,
)

print(retriever_results["documents"])

现在让我们组合最终的提示词：

In [ ]:
# 用户查询
user_query = "S3Eval: A Synthetic, Scalable, Systematic Evaluation Suite for Large Language Models"

# 查询用户查询的相似结果
results = collection.query(
    query_texts=[user_query],
    n_results=10,
)

# 将标题连接成单个字符串
short_titles = '\n'.join(results['documents'][0])

prompt_template = f'''[INST]

Your main task is to generate 5 SUGGESTED_TITLES based for the PAPER_TITLE

You should mimic a similar style and length as SHORT_TITLES but PLEASE DO NOT include titles from SHORT_TITLES in the SUGGESTED_TITLES, only generate versions of the PAPER_TILE.

PAPER_TITLE: {user_query}

SHORT_TITLES: {short_titles}

SUGGESTED_TITLES:

[/INST]
'''

responses = get_completion(prompt_template, model=mistral_llm, max_tokens=2000)
suggested_titles = ''.join([str(r) for r in responses])

# 打印建议
print("Model Suggestions:")
print(suggested_titles)
print("\n\n\nPrompt Template:")
print(prompt_template)

如你所见，LLM 生成的简短标题还算可以。这个用例仍然需要大量工作，可能还需要微调。出于本教程的目的，我们展示了一个使用 Fireworks 闪电般快速的模型进行 RAG 的简单应用。

在这里尝试其他开源模型：https://app.fireworks.ai/models

在此处阅读更多关于 Fireworks API 的信息：https://readme.fireworks.ai/reference/createchatcompletion